# Modelo predictivo-prescriptivo para priorizar el backlog de mantenimiento
## MCDA + optimización ligera con criterios de criticidad

**Autor:** Anibal Antonio De Avila Rueda  
**Programa:** Especialización Tecnológica en Automatización Industrial – Universidad Manuela Beltrán  
**Eje:** Gestión de mantenimiento y operaciones (GIAT)

---
Este notebook orquesta el pipeline completo descrito en el informe avance:
1. Generación de datos sintéticos reproducibles.
2. Entrenamiento + calibración del modelo predictivo.
3. Cálculo del puntaje MCDA.
4. Optimización del plan semanal (MILP / greedy).
5. Comparación contra el baseline tradicional (prioridad/fecha).
6. Generación de KPIs, gráficas y entregables.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_generator import (ConfigEscenario, generar_backlog_semanal,
                              diccionario_datos, features_modelo)
from model      import entrenar_modelo, importancia_variables
from mcda       import calcular_riesgo, PesosMCDA, es_critica
from optimizer  import resolver_plan_semanal
from baseline   import baseline_prioridad_fecha
from evaluation import calcular_kpis, comparar
from scenarios  import escenarios

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

## 2. Generación de datos sintéticos
Backlog semanal entre 20 y 100 WO con atributos de criticidad, historial, uso, inventario y plazos. Se respetan los lineamientos de ISO 14224 (datos de confiabilidad) y RCM (consecuencia y modos de falla).

In [ ]:
cfg = ConfigEscenario()
df_semana = generar_backlog_semanal(semana=1, config=cfg, seed=42)
print(f'Backlog generado: {len(df_semana)} WO')
df_semana.head(8)

In [ ]:
diccionario_datos()

## 3. Entrenamiento y calibración del modelo predictivo
Regresión logística con regularización L2 + balance de clases. Calibración isotónica (PAV) para que las probabilidades reflejen frecuencias observadas.

In [ ]:
semanas_train = [generar_backlog_semanal(s, cfg, seed=7) for s in range(1, 26)]
df_train = pd.concat(semanas_train, ignore_index=True)
print('Dataset de entrenamiento:', df_train.shape)
modelo, met = entrenar_modelo(df_train)
print('Métricas en holdout:', met.como_dict())

In [ ]:
imp = importancia_variables(modelo).head(12)
imp

## 4. Cálculo MCDA
Riesgo = 0.5 · Prob + 0.3 · Severidad + 0.2 · Costo  
Severidad combina seguridad/ambiente/producción con sub-pesos.

In [ ]:
df_test = generar_backlog_semanal(semana=26, config=cfg, seed=99)
proba = modelo.predict_proba(df_test)
df_score = calcular_riesgo(df_test, proba)
df_score[['wo_id', 'criticidad', 'prob_calibrada', 'severidad', 'costo', 'riesgo']].sort_values('riesgo', ascending=False).head(10)

## 5. Resolución del plan semanal
Optimización binaria que maximiza el riesgo programado sujeto a horas totales, horas por habilidad y disponibilidad de repuestos.

In [ ]:
plan_modelo = resolver_plan_semanal(df_score, cfg.horas_totales, cfg.horas_por_habilidad)
plan_baseline = baseline_prioridad_fecha(df_score, cfg.horas_totales, cfg.horas_por_habilidad)
print('MODELO  :', plan_modelo.como_dict())
print('BASELINE:', plan_baseline.como_dict())

## 6. KPIs operativos

In [ ]:
kpi_b = calcular_kpis(df_score, plan_baseline, cfg.horas_totales, cfg.horas_por_habilidad)
kpi_m = calcular_kpis(df_score, plan_modelo,   cfg.horas_totales, cfg.horas_por_habilidad)
comp = comparar(kpi_b, kpi_m)
pd.DataFrame({'baseline': kpi_b.como_dict(), 'modelo': kpi_m.como_dict()}).T

In [ ]:
print('Reducción del riesgo residual: {:.1f}%'.format(comp['reduccion_riesgo_residual_pct']))

## 7. Ejecución completa con 30 réplicas y 5 escenarios
Esto reproduce los resultados reportados en el informe final.  
**Importante:** correr este bloque tarda ~30-60 segundos.

In [ ]:
import subprocess
out = subprocess.run(['python', '../main.py', '--replicas', '30'],
                      capture_output=True, text=True)
print(out.stdout[-2000:])

In [ ]:
df_det = pd.read_csv('../results/resultados_detalle.csv')
rr = (df_det[(df_det['kpi'] == 'riesgo_residual_abs') &
              (df_det['metodo'].isin(['baseline', 'modelo']))]
      .groupby(['escenario', 'metodo'])['valor'].mean().unstack())
rr['reduccion_%'] = 100 * (rr['baseline'] - rr['modelo']) / rr['baseline']
rr.round(2)

## 8. Generación de entregables

In [ ]:
from visuals             import generar_todas
from exportar_excel      import construir as construir_xlsx
from exportar_diccionario import construir as construir_dicc
from exportar_informe    import construir as construir_informe

out_dir = os.path.abspath('../results')
_ = generar_todas(out_dir)
print('Excel  :', construir_xlsx(out_dir, os.path.join(out_dir, 'tabla_resumen.xlsx')))
print('Dicc.  :', construir_dicc(os.path.abspath('../docs/diccionario_datos.docx')))
print('Informe:', construir_informe(out_dir, os.path.join(out_dir, 'informe_final_resultados.docx')))

## 9. Visualización de resultados

In [ ]:
from IPython.display import Image
Image('../results/graficas/barras_reduccion_riesgo.png')

In [ ]:
Image('../results/graficas/boxplot_riesgo_residual.png')

In [ ]:
Image('../results/graficas/calibracion_modelo.png')

---
**Conclusión:** el pipeline integra una predicción calibrada, un puntaje multicriterio explicable y una optimización con restricciones reales. Los resultados muestran reducciones consistentes del riesgo residual frente al baseline, especialmente en escenarios estresados (E2 y E4).